# Tutorial: Drought Bayesian Network IBF V3

This notebook provides a step-by-step educational walkthrough of the drought Bayesian Network analysis system for Impact-Based Forecasting (IBF).

**Key Components:**
- **CDI (Combined Drought Indicator)**: Current drought conditions (0-10 scale)
- **Empirical Probabilities**: Forecast probability of exceeding return period thresholds
- **Bayesian Network**: Probabilistic inference for risk assessment

**Author:** ICPAC Drought IBF Team  
**Version:** 3.0

---
## Step 1: Imports and Setup

In [ ]:
import numpy as np
import pandas as pd
import xarray as xr
import geopandas as gpd
import matplotlib.pyplot as plt
import networkx as nx
from matplotlib.patches import Patch

# Set display options
pd.set_option('display.max_columns', 20)
plt.rcParams['figure.dpi'] = 100

print("Imports successful!")

In [ ]:
# Import from the drought module
from drought_bn_ibf_v3 import (
    DroughtDataLoaderV3,
    DroughtBayesianNetworkV3,
    CDIDataLoader,
    categorize_cdi,
    get_season_from_lead,
    analyze_drought
)

print("Drought BN V3 module loaded successfully!")

---
## Step 2: Define Data Paths and Parameters

In [ ]:
# Define paths to your data
BOUNDARIES_PATH = "icpac_adm1v3.geojson"
EPROB_PATH = "/srv/empirical_probability_output/empirical_probabilities.nc"
CDI_BASE_PATH = "/srv/icpac_monthly_netcdf"

# Forecast parameters
INIT_YEAR = 2024
INIT_MONTH = 12
TARGET_LEAD = 5  # Lead time in months (0-indexed, so 5 = 6th month)

print(f"Configuration:")
print(f"  Boundaries: {BOUNDARIES_PATH}")
print(f"  Empirical Probs: {EPROB_PATH}")
print(f"  CDI Base: {CDI_BASE_PATH}")
print(f"  Init: {INIT_YEAR}-{INIT_MONTH:02d}")
print(f"  Target Lead: {TARGET_LEAD}")

---
## Step 3: Load and Explore Admin Boundaries

In [ ]:
# Load boundaries manually to explore
boundaries = gpd.read_file(BOUNDARIES_PATH)

print(f"Total boundaries: {len(boundaries)}")
print(f"\nColumns: {boundaries.columns.tolist()}")
print(f"\nCRS: {boundaries.crs}")

boundaries.head()

In [ ]:
# Visualize the boundaries
fig, ax = plt.subplots(figsize=(12, 10))
boundaries.plot(ax=ax, edgecolor='black', facecolor='lightblue', alpha=0.5, linewidth=0.5)
ax.set_title("ICPAC Admin Level 1 Boundaries", fontsize=14)
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
plt.tight_layout()
plt.show()

In [ ]:
# Count boundaries by country
if 'GID_1' in boundaries.columns:
    boundaries['country_code'] = boundaries['GID_1'].str.split('.').str[0]
    print("Boundaries by Country:")
    print(boundaries['country_code'].value_counts())

---
## Step 4: Initialize the Data Loader

In [ ]:
# Create the data loader - this loads boundaries and prepares for analysis
loader = DroughtDataLoaderV3(
    boundaries_path=BOUNDARIES_PATH,
    eprob_path=EPROB_PATH,
    cdi_base_path=CDI_BASE_PATH
)

print(f"\nLoader initialized with {loader.n_boundaries} boundaries")
print(f"Countries: {sorted(loader.boundaries['country'].unique())}")

---
## Step 5: Explore Empirical Probability (Forecast) Data

In [ ]:
# Load and explore the forecast probability data
eprob_ds = xr.open_dataset(EPROB_PATH)
print("Empirical Probability Dataset:")
print(eprob_ds)

In [ ]:
# See available return period variables
rp_vars = [v for v in eprob_ds.data_vars if 'eprob' in v]
print(f"Return period variables: {rp_vars}")

# Check dimensions
print(f"\nInit times available: {len(eprob_ds.init)} total")
print(f"First 5 init times: {eprob_ds.init.values[:5]}")
print(f"\nLead times: {eprob_ds.lead.values}")
print(f"\nGrid shape: {len(eprob_ds.lat)} lat x {len(eprob_ds.lon)} lon")

In [ ]:
# Select specific init time and lead
init_time = f"{INIT_YEAR}-{INIT_MONTH:02d}-01"
season = get_season_from_lead(INIT_MONTH, TARGET_LEAD)

print(f"Selected initialization: {init_time}")
print(f"Target lead: {TARGET_LEAD} months")
print(f"Target season: {season} SPI3")

# Subset the data
eprob_subset = eprob_ds.sel(init=init_time, method='nearest').sel(lead=TARGET_LEAD)
print(f"\nSubset dimensions: {dict(eprob_subset.sizes)}")

In [ ]:
# Plot the 5-year return period probability
fig, ax = plt.subplots(figsize=(14, 10))

eprob_subset['eprob_5yr'].plot(
    ax=ax, 
    cmap='YlOrRd', 
    vmin=0, 
    vmax=1,
    cbar_kwargs={'label': 'Exceedance Probability'}
)

# Overlay boundaries
loader.boundaries.boundary.plot(ax=ax, color='black', linewidth=0.5)

ax.set_title(f"5-Year Return Period Exceedance Probability\n{season} SPI3 (Init: {init_time}, Lead: {TARGET_LEAD})", fontsize=14)
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
plt.tight_layout()
plt.show()

In [ ]:
# Compare multiple return periods
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for ax, rp_var, title in zip(axes, ['eprob_5yr', 'eprob_10yr', 'eprob_20yr'], 
                              ['5-Year RP', '10-Year RP', '20-Year RP']):
    eprob_subset[rp_var].plot(ax=ax, cmap='YlOrRd', vmin=0, vmax=1, add_colorbar=True)
    loader.boundaries.boundary.plot(ax=ax, color='black', linewidth=0.3)
    ax.set_title(f"{title} Exceedance Probability")
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")

plt.suptitle(f"{season} SPI3 Forecast (Init: {init_time})", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

---
## Step 6: Explore CDI (Combined Drought Indicator) Data

In [ ]:
# CDI categories explanation
print("CDI (Combined Drought Indicator) Categories:")
print("="*50)
categories = {
    '0': 'No Drought - Normal or wet conditions',
    '1-2': 'Mild - Watch conditions',
    '3-4': 'Moderate - Warning conditions', 
    '5-7': 'Severe - Alert conditions',
    '8-10': 'Extreme - Emergency conditions'
}
for value, desc in categories.items():
    print(f"  CDI {value:5}: {desc}")

In [ ]:
# Check CDI availability
cdi_loader = CDIDataLoader(CDI_BASE_PATH)

# Find latest available CDI relative to our init time
cdi_year, cdi_month, lag = cdi_loader.get_latest_available(INIT_YEAR, INIT_MONTH)
print(f"Latest available CDI: {cdi_year}-{cdi_month:02d}")
print(f"Lag from init time: {lag} months")

In [ ]:
# Load CDI data
cdi = cdi_loader.load_cdi(cdi_year, cdi_month)
print(f"CDI shape: {cdi.shape}")
print(f"CDI range: {float(cdi.min()):.2f} to {float(cdi.max()):.2f}")

In [ ]:
# Plot CDI
fig, ax = plt.subplots(figsize=(14, 10))

# Custom colormap for CDI
from matplotlib.colors import ListedColormap, BoundaryNorm
cdi_colors = ['#FFFFFF', '#FFFF00', '#FCD37F', '#FFAA00', '#E60000', '#730000']
cdi_bounds = [0, 1, 3, 5, 7, 9, 11]
cdi_cmap = ListedColormap(cdi_colors)
cdi_norm = BoundaryNorm(cdi_bounds, cdi_cmap.N)

im = ax.imshow(cdi.values, cmap=cdi_cmap, norm=cdi_norm, 
               extent=[21.91, 51.40, -11.69, 23.10], origin='upper')

cbar = plt.colorbar(im, ax=ax, ticks=[0.5, 2, 4, 6, 8, 10])
cbar.set_ticklabels(['No Drought', 'Mild', 'Moderate', 'Severe', 'Extreme', ''])
cbar.set_label('CDI Category')

ax.set_title(f"Combined Drought Indicator (CDI)\n{cdi_year}-{cdi_month:02d}", fontsize=14)
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
plt.tight_layout()
plt.show()

---
## Step 7: Prepare Analysis Data for All Boundaries

This step extracts data for all admin boundaries using:
- **Area-based extraction** for large boundaries (mean of pixels within boundary)
- **Centroid-based extraction** for small boundaries (nearest grid point to centroid)

In [ ]:
# Prepare analysis data for all boundaries
boundaries_data, metadata = loader.prepare_analysis_data(
    init_year=INIT_YEAR,
    init_month=INIT_MONTH,
    target_lead=TARGET_LEAD,
    cdi_scenario='actual'  # Options: 'actual', 'perfect', 'lag_2', 'lag_3'
)

In [ ]:
# Display metadata
print("Analysis Metadata:")
print("="*50)
for key, value in metadata.items():
    print(f"  {key}: {value}")

In [ ]:
# Inspect one boundary's data
sample_idx = 50  # Change to explore different boundaries
sample = boundaries_data[sample_idx]

print(f"\nSample Boundary Data (index {sample_idx}):")
print("="*50)
for key, value in sample.items():
    if isinstance(value, float):
        print(f"  {key:20}: {value:.4f}")
    else:
        print(f"  {key:20}: {value}")

In [ ]:
# Convert to DataFrame for easier exploration
boundaries_df = pd.DataFrame(boundaries_data)
print(f"Boundaries DataFrame shape: {boundaries_df.shape}")
boundaries_df.head(10)

In [ ]:
# Summary statistics
print("Extraction Method Distribution:")
print(boundaries_df['extraction_method'].value_counts())

print("\nCDI Category Distribution:")
print(boundaries_df['cdi_category'].value_counts())

print("\nSeverity Index Statistics:")
print(boundaries_df['severity_index'].describe())

---
## Step 8: Initialize and Visualize the Bayesian Network

The Bayesian Network structure:
```
antecedent_condition ---+
                        |
exceedance_prob --------+---> risk_level ---> action
                        |
spatial_coverage -------+
```

In [ ]:
# Create the Bayesian Network
bn = DroughtBayesianNetworkV3()

print("Bayesian Network Structure:")
print("="*50)
print(f"Nodes: {list(bn.model.nodes())}")
print(f"Edges: {list(bn.model.edges())}")
print(f"\nCPT Statistics:")
for key, value in bn.cpt_stats.items():
    print(f"  {key}: {value}")

In [ ]:
# Visualize the DAG structure
fig, ax = plt.subplots(figsize=(14, 10))

# Create NetworkX graph from BN edges
G = nx.DiGraph()
G.add_edges_from(bn.model.edges())

# Define positions for a clean layout
pos = {
    'antecedent_condition': (0, 1),
    'exceedance_prob': (1, 1),
    'spatial_coverage': (2, 1),
    'risk_level': (1, 0.5),
    'action': (1, 0)
}

# Node colors by type
node_colors = {
    'antecedent_condition': '#90EE90',  # Light green - observed (CDI)
    'exceedance_prob': '#87CEEB',       # Sky blue - observed (forecast)
    'spatial_coverage': '#DDA0DD',      # Plum - observed (spatial)
    'risk_level': '#FFD700',            # Gold - intermediate (inferred)
    'action': '#FF6347'                 # Tomato - decision (output)
}

colors = [node_colors[node] for node in G.nodes()]

# Draw the network
nx.draw(G, pos, ax=ax, with_labels=True, node_color=colors,
        node_size=5000, font_size=11, font_weight='bold',
        arrows=True, arrowsize=25, edge_color='gray', width=2,
        connectionstyle='arc3,rad=0.1')

ax.set_title("Drought Risk Bayesian Network DAG\n(V3 - Simplified Structure)", fontsize=16)

# Add legend
legend_elements = [
    Patch(facecolor='#90EE90', label='Antecedent Condition (CDI)', edgecolor='black'),
    Patch(facecolor='#87CEEB', label='Exceedance Probability (Forecast)', edgecolor='black'),
    Patch(facecolor='#DDA0DD', label='Spatial Coverage', edgecolor='black'),
    Patch(facecolor='#FFD700', label='Risk Level (Inferred)', edgecolor='black'),
    Patch(facecolor='#FF6347', label='Action (Output)', edgecolor='black')
]
ax.legend(handles=legend_elements, loc='upper right', fontsize=10)

plt.tight_layout()
plt.show()

In [ ]:
# Display node state descriptions
print("Node States:")
print("="*60)

print("\n1. antecedent_condition (from CDI):")
print("   No_Drought, Mild, Moderate, Severe, Extreme")

print("\n2. exceedance_prob (from severity_index):")
print("   Very_Low (<0.2), Low (0.2-0.4), Medium (0.4-0.6), High (0.6-0.8), Very_High (>0.8)")

print("\n3. spatial_coverage (% of area with high probability):")
print("   Localized (<30%), Moderate (30-60%), Widespread (>60%)")

print("\n4. risk_level (inferred):")
print("   Minimal, Low, Moderate, High, Extreme")

print("\n5. action (recommended):")
print("   Monitor, Be_Aware, Be_Prepared, Take_Action")

---
## Step 9: Explore Conditional Probability Distributions (CPDs)

In [ ]:
# Examine the action CPD
action_cpd = bn.model.get_cpds('action')
print("Action CPD given Risk Level:")
print(action_cpd)

In [ ]:
# Create a heatmap of Action probabilities
action_probs = action_cpd.values
risk_states = ['Minimal', 'Low', 'Moderate', 'High', 'Extreme']
action_states = ['Monitor', 'Be_Aware', 'Be_Prepared', 'Take_Action']

fig, ax = plt.subplots(figsize=(12, 6))
im = ax.imshow(action_probs, cmap='YlOrRd', aspect='auto', vmin=0, vmax=1)

ax.set_xticks(range(len(risk_states)))
ax.set_xticklabels(risk_states, fontsize=12)
ax.set_yticks(range(len(action_states)))
ax.set_yticklabels(action_states, fontsize=12)

ax.set_xlabel('Risk Level', fontsize=14)
ax.set_ylabel('Recommended Action', fontsize=14)
ax.set_title('P(Action | Risk Level)', fontsize=16)

# Add probability values as text
for i in range(len(action_states)):
    for j in range(len(risk_states)):
        color = 'white' if action_probs[i, j] > 0.5 else 'black'
        ax.text(j, i, f'{action_probs[i, j]:.2f}', ha='center', va='center', 
                fontsize=14, fontweight='bold', color=color)

plt.colorbar(im, label='Probability')
plt.tight_layout()
plt.show()

In [ ]:
# Examine prior CPDs
print("Prior CPDs (marginal probabilities):")
print("="*50)

for node in ['antecedent_condition', 'exceedance_prob', 'spatial_coverage']:
    cpd = bn.model.get_cpds(node)
    print(f"\n{node}:")
    states = cpd.state_names[node]
    probs = cpd.values.flatten()
    for state, prob in zip(states, probs):
        print(f"  {state:15}: {prob:.2f}")

---
## Step 10: Process a Single Boundary (Step-by-Step Inference)

In [ ]:
# Pick a boundary to analyze
sample_boundary = boundaries_data[50]  # Change index to explore others

print(f"Analyzing: {sample_boundary['name']} ({sample_boundary['country']})")
print("="*60)
print(f"  CDI Mean: {sample_boundary['cdi_mean']:.2f}")
print(f"  CDI Category: {sample_boundary['cdi_category']}")
print(f"  Severity Index: {sample_boundary['severity_index']:.3f}")
print(f"  Spatial Coverage: {sample_boundary['spatial_coverage']*100:.1f}%")
print(f"\n  Return Period Probabilities:")
print(f"    3-year: {sample_boundary.get('eprob_3yr', 0):.3f}")
print(f"    5-year: {sample_boundary.get('eprob_5yr', 0):.3f}")
print(f"   10-year: {sample_boundary.get('eprob_10yr', 0):.3f}")
print(f"   20-year: {sample_boundary.get('eprob_20yr', 0):.3f}")
print(f"   50-year: {sample_boundary.get('eprob_50yr', 0):.3f}")

In [ ]:
# See how continuous values are discretized into evidence states
antecedent = bn._categorize_antecedent(sample_boundary['cdi_category'])
exceedance = bn._categorize_exceedance(sample_boundary['severity_index'])
spatial = bn._categorize_spatial(sample_boundary['spatial_coverage'])

print("\nDiscretized Evidence States for BN:")
print("="*50)
print(f"  antecedent_condition: '{antecedent}'")
print(f"  exceedance_prob: '{exceedance}'")
print(f"  spatial_coverage: '{spatial}'")

In [ ]:
# Run Bayesian inference
from pgmpy.inference import VariableElimination

inference = VariableElimination(bn.model)

evidence = {
    'antecedent_condition': antecedent,
    'exceedance_prob': exceedance,
    'spatial_coverage': spatial
}

print("\nEvidence provided:")
for k, v in evidence.items():
    print(f"  {k} = {v}")

In [ ]:
# Query risk level
risk_result = inference.query(variables=['risk_level'], evidence=evidence)

print("\nRisk Level Posterior Probabilities:")
print("="*50)
for state, prob in zip(risk_result.state_names['risk_level'], risk_result.values):
    bar = '█' * int(prob * 40)
    print(f"  {state:10}: {prob:.3f} {bar}")

most_likely_risk = risk_result.state_names['risk_level'][np.argmax(risk_result.values)]
print(f"\n  Most likely risk level: {most_likely_risk}")

In [ ]:
# Query action
action_result = inference.query(variables=['action'], evidence=evidence)

print("\nAction Posterior Probabilities:")
print("="*50)
for state, prob in zip(action_result.state_names['action'], action_result.values):
    bar = '█' * int(prob * 40)
    print(f"  {state:12}: {prob:.3f} {bar}")

recommended_action = action_result.state_names['action'][np.argmax(action_result.values)]
confidence = np.max(action_result.values)
print(f"\n  Recommended action: {recommended_action} (confidence: {confidence:.1%})")

In [ ]:
# Visualize the inference results
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Risk Level
ax1 = axes[0]
risk_colors = ['#2ecc71', '#82e0aa', '#f7dc6f', '#e67e22', '#e74c3c']
bars1 = ax1.barh(risk_result.state_names['risk_level'], risk_result.values, color=risk_colors)
ax1.set_xlabel('Probability', fontsize=12)
ax1.set_title('Risk Level Distribution', fontsize=14)
ax1.set_xlim(0, 1)
for bar, prob in zip(bars1, risk_result.values):
    ax1.text(prob + 0.02, bar.get_y() + bar.get_height()/2, f'{prob:.2f}', va='center')

# Action
ax2 = axes[1]
action_colors = ['#2ecc71', '#f1c40f', '#e67e22', '#e74c3c']
bars2 = ax2.barh(action_result.state_names['action'], action_result.values, color=action_colors)
ax2.set_xlabel('Probability', fontsize=12)
ax2.set_title('Action Distribution', fontsize=14)
ax2.set_xlim(0, 1)
for bar, prob in zip(bars2, action_result.values):
    ax2.text(prob + 0.02, bar.get_y() + bar.get_height()/2, f'{prob:.2f}', va='center')

plt.suptitle(f"{sample_boundary['name']} ({sample_boundary['country']})", fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

---
## Step 11: Process All Boundaries

In [ ]:
# Process all boundaries through the BN
print("Processing all boundaries through Bayesian Network...")
import time

start_time = time.time()
results_df = bn.process_all_boundaries(boundaries_data)
elapsed = time.time() - start_time

print(f"\nProcessed {len(results_df)} boundaries in {elapsed:.2f} seconds")

In [ ]:
# View results
results_df.head(10)

In [ ]:
# Summary statistics
print("Results Summary:")
print("="*60)

print("\nAction Distribution:")
action_dist = results_df['recommended_action'].value_counts()
for action in ['Monitor', 'Be_Aware', 'Be_Prepared', 'Take_Action']:
    count = action_dist.get(action, 0)
    pct = count / len(results_df) * 100
    print(f"  {action:15}: {count:3} ({pct:.1f}%)")

print("\nRisk Level Distribution:")
print(results_df['risk_level'].value_counts())

In [ ]:
# High risk boundaries by country
high_risk = results_df[results_df['recommended_action'].isin(['Be_Prepared', 'Take_Action'])]

print(f"\nHigh Risk Boundaries: {len(high_risk)} total")
print("\nBy Country:")
if len(high_risk) > 0:
    print(high_risk['country'].value_counts())
else:
    print("  No high-risk boundaries identified")

In [ ]:
# Top 15 highest severity boundaries
print("\nTop 15 Highest Severity Boundaries:")
print("="*80)
top_15 = results_df.nlargest(15, 'severity_index')[[
    'boundary_name', 'country', 'cdi_category', 'severity_index', 
    'risk_level', 'recommended_action', 'confidence'
]]
print(top_15.to_string(index=False))

---
## Step 12: Visualize Results on Map

In [ ]:
# Merge results with boundaries for mapping
results_geo = loader.boundaries.merge(
    results_df[['boundary_id', 'recommended_action', 'risk_level', 'severity_index']],
    left_on='id', right_on='boundary_id'
)

print(f"Merged GeoDataFrame: {len(results_geo)} boundaries")

In [ ]:
# Define action colors
action_colors = {
    'Monitor': '#2ecc71',      # Green
    'Be_Aware': '#f1c40f',     # Yellow
    'Be_Prepared': '#e67e22',  # Orange
    'Take_Action': '#e74c3c'   # Red
}

results_geo['color'] = results_geo['recommended_action'].map(action_colors)

# Plot recommended actions
fig, ax = plt.subplots(figsize=(16, 12))

results_geo.plot(ax=ax, color=results_geo['color'], edgecolor='black', linewidth=0.3)

ax.set_title(f"Drought Risk Assessment - {metadata['target_season']} SPI3\n"
             f"Init: {metadata['init_time']}, Lead: {metadata['target_lead']} months", fontsize=16)
ax.set_xlabel("Longitude", fontsize=12)
ax.set_ylabel("Latitude", fontsize=12)

# Legend
legend_elements = [Patch(facecolor=c, label=a, edgecolor='black') 
                   for a, c in action_colors.items()]
ax.legend(handles=legend_elements, loc='lower left', title='Recommended Action', fontsize=11)

plt.tight_layout()
plt.show()

In [ ]:
# Plot severity index (continuous)
fig, ax = plt.subplots(figsize=(16, 12))

results_geo.plot(ax=ax, column='severity_index', cmap='YlOrRd', 
                 edgecolor='black', linewidth=0.3, legend=True,
                 legend_kwds={'label': 'Severity Index', 'shrink': 0.6})

ax.set_title(f"Drought Severity Index - {metadata['target_season']} SPI3\n"
             f"Init: {metadata['init_time']}, Lead: {metadata['target_lead']} months", fontsize=16)
ax.set_xlabel("Longitude", fontsize=12)
ax.set_ylabel("Latitude", fontsize=12)

plt.tight_layout()
plt.show()

---
## Step 13: Quick Analysis Using Convenience Function

For production use, the `analyze_drought()` function wraps all steps into one call.

In [ ]:
# Quick analysis - all in one function call
results = analyze_drought(
    boundaries_path=BOUNDARIES_PATH,
    eprob_path=EPROB_PATH,
    cdi_base_path=CDI_BASE_PATH,
    init_year=INIT_YEAR,
    init_month=INIT_MONTH,
    target_lead=TARGET_LEAD,
    cdi_scenario='actual',
    output_path='drought_bn_v3_tutorial_results.csv'
)

---
## Step 14: Sensitivity Analysis (Optional)

Explore how different evidence states affect the output.

In [ ]:
# Sensitivity analysis: How does action change with different exceedance probabilities?
from pgmpy.inference import VariableElimination

inference = VariableElimination(bn.model)

# Fix antecedent and spatial, vary exceedance
antecedent_fixed = 'Moderate'
spatial_fixed = 'Moderate'
exceedance_states = ['Very_Low', 'Low', 'Medium', 'High', 'Very_High']

print(f"Sensitivity Analysis: Varying Exceedance Probability")
print(f"Fixed: antecedent_condition={antecedent_fixed}, spatial_coverage={spatial_fixed}")
print("="*70)

sensitivity_results = []
for exc in exceedance_states:
    evidence = {
        'antecedent_condition': antecedent_fixed,
        'exceedance_prob': exc,
        'spatial_coverage': spatial_fixed
    }
    action_result = inference.query(variables=['action'], evidence=evidence)
    action_probs = dict(zip(action_result.state_names['action'], action_result.values))
    action_probs['exceedance'] = exc
    sensitivity_results.append(action_probs)

sensitivity_df = pd.DataFrame(sensitivity_results)
sensitivity_df = sensitivity_df.set_index('exceedance')
print(sensitivity_df.round(3))

In [ ]:
# Visualize sensitivity
fig, ax = plt.subplots(figsize=(12, 6))

sensitivity_df.plot(kind='bar', ax=ax, color=['#2ecc71', '#f1c40f', '#e67e22', '#e74c3c'])
ax.set_xlabel('Exceedance Probability State', fontsize=12)
ax.set_ylabel('Action Probability', fontsize=12)
ax.set_title(f'Sensitivity: Action vs Exceedance\n(Antecedent={antecedent_fixed}, Spatial={spatial_fixed})', fontsize=14)
ax.legend(title='Action', bbox_to_anchor=(1.02, 1))
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
plt.tight_layout()
plt.show()

---
## Summary: Key Concepts

| Component | Description |
|-----------|-------------|
| **CDI** | Combined Drought Indicator (0-10) - current drought conditions |
| **eprob_Xyr** | Empirical probability of exceeding X-year return period SPI3 |
| **severity_index** | Combined metric from multiple return periods |
| **spatial_coverage** | Fraction of boundary area with high exceedance probability |
| **risk_level** | BN-inferred risk: Minimal → Extreme |
| **action** | Recommended response: Monitor → Take_Action |

### Action Interpretation:
- **Monitor** (Green): Normal conditions, routine monitoring
- **Be_Aware** (Yellow): Elevated risk, increase monitoring frequency
- **Be_Prepared** (Orange): Prepare contingency plans, pre-position resources
- **Take_Action** (Red): Activate response plans, implement interventions

In [ ]:
print("Tutorial completed!")
print("\nOutput files:")
print("  - drought_bn_v3_tutorial_results.csv")